In [ ]:
#!/usr/bin/env python3
"""
vllm_pipeline_v5_2.py -- Neuro-Symbolic Pipeline (Best-of-N + SpaCy)

EXACT 2026 -- XAI Challenge @ IJCNN
Qwen2.5-7B + Z3 + vLLM | Kaggle T4x2, No Cloud API

Pipeline v5.2 -- Best-of-N + SpaCy Entity Extraction:
  Stage 0: Install & Config
  Stage 1: Load vLLM Engine
  Stage 2: Prompts (Pass 1 & Pass 2)
  Stage 3: Z3 Compiler & Entailment Checker
  Stage 4: Batch Pipeline (Pass 1: Premises -> Z3 -> Pass 2: Questions -> Z3 Entailment)
  Stage 5: Evaluation & Export

Key improvements over v5.1:
  - Splits formalization into 2 passes to avoid LLM overload & VRAM OOM crashes.
  - Pass 1: Formalize Premises (high SAT rate).
  - Pass 2: Formalize Questions (using proven Ontology).
"""


In [ ]:
# ==================================================================
# FIX cho Kaggle T4 -- CELL ĐẦU TIÊN, CHẠY TRƯỚC TẤT CẢ
# ==================================================================
import os, shutil, glob

# --- Fix 1: Symlink libcuda.so ---
STUB_DIR = "/tmp/cuda_stubs"
os.makedirs(STUB_DIR, exist_ok=True)
stub = os.path.join(STUB_DIR, "libcuda.so")

# Xóa link cũ
if os.path.exists(stub) or os.path.islink(stub):
    os.remove(stub)

# Tìm libcuda.so.1 thực tế trên Kaggle
for candidate in [
    "/usr/lib/x86_64-linux-gnu/libcuda.so.1",
    "/usr/lib/x86_64-linux-gnu/libcuda.so",
    *glob.glob("/usr/**/libcuda.so*", recursive=True),
]:
    if os.path.exists(candidate) and not os.path.islink(candidate):
        os.symlink(candidate, stub)
        print(f"✓ Symlink: {stub} -> {candidate}")
        break
else:
    # Fallback: tìm bằng ldconfig
    import subprocess
    result = subprocess.run(["ldconfig", "-p"], capture_output=True, text=True)
    for line in result.stdout.splitlines():
        if "libcuda.so" in line:
            path = line.split("=>")[-1].strip()
            if os.path.exists(path):
                os.symlink(path, stub)
                print(f"✓ Symlink (ldconfig): {stub} -> {path}")
                break

# --- Fix 2: QUAN TRỌNG - Set CẢ compile-time VÀ runtime path ---
os.environ["LIBRARY_PATH"] = f"{STUB_DIR}:" + os.environ.get("LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = f"{STUB_DIR}:" + os.environ.get("LD_LIBRARY_PATH", "")

# --- Fix 3: Xóa cache FlashInfer bị lỗi ---
shutil.rmtree("/root/.cache/flashinfer", ignore_errors=True)

# --- Fix 4: Verify ---
print(f"LIBRARY_PATH    = {os.environ.get('LIBRARY_PATH', '')[:80]}...")
print(f"LD_LIBRARY_PATH = {os.environ.get('LD_LIBRARY_PATH', '')[:80]}...")
print(f"Stub exists     = {os.path.exists(stub)}")
if os.path.exists(stub):
    print(f"Stub target     = {os.path.realpath(stub)}")
print("\n✓ Kaggle T4 fixes applied!")

In [ ]:

# ==================================================================
# STAGE 0 -- Install Dependencies & Config
# ==================================================================

import subprocess, sys

pkgs = [
    "vllm",
    "z3-solver",
    "spacy",
]
print("Installing vLLM + z3-solver (co the mat 2-5 phut)...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--break-system-packages"]
    + pkgs,
    check=True,
)
print("All packages installed OK")

import json, os, re, time, csv, traceback
from pathlib import Path
from dataclasses import dataclass, field

import torch
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from z3 import (
    Int, IntSort, IntVal, BoolSort, Function,
    ForAll, Exists, And, Or, Not, Implies, Solver, sat, unsat,
)

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA OK  : {torch.cuda.is_available()}")
N_GPUS = torch.cuda.device_count()
for i in range(N_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}    : {props.name}  ({props.total_memory / 1024**3:.1f} GB)")
print("Imports OK")

# --- SpaCy Setup ---
import spacy
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")
print(f"SpaCy model: {nlp.meta['name']} v{nlp.meta['version']}")
print("SpaCy OK")


In [ ]:

# ==================================================================
# CAU HINH -- Chinh sua o day
# ==================================================================
QWEN_MODEL_ID  = "Qwen/Qwen2.5-7B-Instruct"
DATASET_PATH   = "C:/Users/Minh Triet/Downloads/Logic_Based_Educational_Queries (2).json"
PHYSICS_DATASET_PATH = "C:/Users/Minh Triet/Downloads/Physics_Problems.csv"

# --- Data Split Config ---
SEED = 42
TRAIN_RATIO = 0.85
VAL_RATIO   = 0.10
TEST_RATIO  = 0.05
RUN_ON_SPLIT = "test"  # 'test', 'train', 'val', hoac 'all'

N_SAMPLES      = 411
# --- Best-of-N Config ---
BEST_OF_N       = 5     # Number of candidates per sample
BON_TEMPERATURE = 0.6   # Higher temp for diversity

MAX_RETRIES    = 2               # Giam xuong 2 vi v4 nhe hon
OUTPUT_PATH    = "/kaggle/working/pipeline_results_v5_2.json"
MAX_NEW_TOKENS_PASS1 = 4096      # Token cho Premises
MAX_NEW_TOKENS_PASS2 = 1024      # Token cho Question (it hon)
ANS_MAX_TOKENS       = 600       # Token cho Qwen Fallback

# vLLM Config
TENSOR_PARALLEL  = min(N_GPUS, 2)
MAX_MODEL_LEN    = 8192
GPU_MEMORY_UTIL  = 0.85
DTYPE            = "half"

# Physics Config
PHYSICS_MODE       = "direct"
PHYSICS_MAX_TOKENS = 1024
PHYSICS_TOLERANCE  = 0.05

# Z3 Entailment Config
Z3_ENTAILMENT      = True
Z3_SOLVER_TIMEOUT  = 5000
# ==================================================================

print(f"Config OK - V4 TWO-PASS")


In [ ]:

# ==================================================================
# STAGE 1 -- Load vLLM Engine
# ==================================================================

print(f"\nLoading vLLM engine ({QWEN_MODEL_ID})...")
print("  (Lan dau tai ~15 GB tu HuggingFace, sau do cache)")

t_load = time.time()

llm = LLM(
    model=QWEN_MODEL_ID,
    tensor_parallel_size=TENSOR_PARALLEL,
    dtype=DTYPE,
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEMORY_UTIL,
    trust_remote_code=True,
    enforce_eager=False,       # Dung CUDA graph de tang toc
)

tokenizer = AutoTokenizer.from_pretrained(
    QWEN_MODEL_ID, trust_remote_code=True
)

t_loaded = time.time() - t_load
print(f"vLLM Engine loaded in {t_loaded:.1f}s")
print("OK")



In [ ]:

# ==================================================================
# STAGE 2 -- Ontology & Prompts (Two-Pass)
# ==================================================================

GLOBAL_ONTOLOGY_TEXT = """
## GLOBAL ONTOLOGY -- BAT BUOC TUAN THU

### Quantifiers:
  forall -> forall  |  exists -> exists

### Logical Operators:
  and, or, implies, iff, not

### AST Node Types (4 loai):
  quantifier : { "type":"quantifier",  "operator":"forall|exists",
                 "bound_variables":["x",...], "body":{...} }
  connective : { "type":"connective",  "operator":"and|or|implies|iff|not",
                 "operands":[{...},...] }
  predicate  : { "type":"predicate",   "name":"PredicateName",
                 "arguments":["x","y",...] }
  variable   : { "type":"variable",    "name":"x" }
  constant   : { "type":"constant",    "name":"SomeName" }

### QUY TAC:
  1. Chi dung 4 node types tren
  2. 'not' chi co DUNG 1 operand
  3. 'implies' co DUNG 2 operands
  4. bound_variables phai la list
  5. Variables: lowercase (x,y,z), Constants: PascalCase
"""

# ------------------------------------------------------------------
# FEW-SHOT EXAMPLES (CRITICAL FOR QWEN-7B)
# ------------------------------------------------------------------

FEW_SHOT_PREMISES = """
### VÍ DỤ HOÀN CHỈNH:

Premises:
  "All students who study hard pass the exam."
  "Alice is a student."
  "Alice studies hard."

Output:
{
  "step1_local_ontology": [
    {"predicate": "Student",   "arity": 1, "description": "x is a student"},
    {"predicate": "StudyHard", "arity": 1, "description": "x studies hard"},
    {"predicate": "PassExam",  "arity": 1, "description": "x passes the exam"}
  ],
  "step2_premises_ast": [
    {
      "premise_id": 0,
      "source_nl": "All students who study hard pass the exam.",
      "ast": {
        "type": "quantifier", "operator": "forall",
        "bound_variables": ["x"],
        "body": {
          "type": "connective", "operator": "implies",
          "operands": [
            {"type": "connective", "operator": "and", "operands": [
              {"type": "predicate", "name": "Student",   "arguments": ["x"]},
              {"type": "predicate", "name": "StudyHard", "arguments": ["x"]}
            ]},
            {"type": "predicate", "name": "PassExam", "arguments": ["x"]}
          ]
        }
      }
    },
    { "premise_id": 1, "source_nl": "Alice is a student.",
      "ast": {"type": "predicate", "name": "Student", "arguments": ["Alice"]} },
    { "premise_id": 2, "source_nl": "Alice studies hard.",
      "ast": {"type": "predicate", "name": "StudyHard", "arguments": ["Alice"]} }
  ]
}
"""

FEW_SHOT_QUESTIONS = """
### VÍ DỤ HOÀN CHỈNH:
Question 1: "Is it true that Alice passes the exam?" (Yes/No)
Question 2: "Which is correct? A. Alice fails. B. Alice passes." (Multiple Choice)

Output:
{
  "step3_question_fol": [
    {
      "question_id": 0,
      "question_type": "yes_no",
      "statement_ast": {"type": "predicate", "name": "PassExam", "arguments": ["Alice"]}
    },
    {
      "question_id": 1,
      "question_type": "multiple_choice",
      "options_ast": {
         "A": {"type": "connective", "operator": "not", "operands": [{"type": "predicate", "name": "PassExam", "arguments": ["Alice"]}]},
         "B": {"type": "predicate", "name": "PassExam", "arguments": ["Alice"]}
      }
    }
  ]
}
"""

# ------------------------------------------------------------------
# PASS 1 PROMPTS (PREMISES ONLY)
# ------------------------------------------------------------------

PREMISE_FORMALIZATION_SYSTEM = (
    "Ban la chuyen gia logic hinh thuc (FOL). Nhiem vu:\n"
    "  Buoc 1: Tao LOCAL ONTOLOGY -- danh sach Predicates\n"
    "  Buoc 2: Chuyen TUNG tien de thanh cay AST JSON de quy\n\n"
    + GLOBAL_ONTOLOGY_TEXT + "\n"
    + FEW_SHOT_PREMISES + "\n"
    "Output JSON THUAN TUY (khong markdown, khong text thua):\n"
    '{\n'
    '  "step1_local_ontology": [\n'
    '    {"predicate": "Name", "arity": N, "description": "..."}\n'
    '  ],\n'
    '  "step2_premises_ast": [\n'
    '    {"premise_id": 0, "source_nl": "...", "ast": { <AST> }}\n'
    '  ]\n'
    '}\n'
)

CORRECTION_SYSTEM = (
    "Ban la chuyen gia sua loi FOL. He thong Z3 da phat hien loi.\n"
    "Nhiem vu: sua lai Buoc 1 + Buoc 2 de khong con loi.\n\n"
    + GLOBAL_ONTOLOGY_TEXT + "\n"
    "Output JSON thuan tuy -- format GIONG HET lan dau.\n"
)

# ------------------------------------------------------------------
# PASS 2 PROMPTS (QUESTIONS ONLY)
# ------------------------------------------------------------------

QUESTION_FORMALIZATION_SYSTEM = (
    "Ban la chuyen gia logic hinh thuc (FOL). Nhiem vu:\n"
    "  Chuyen TUNG cau hoi thanh AST JSON de kiem tra Z3 Entailment.\n\n"
    + GLOBAL_ONTOLOGY_TEXT + "\n"
    + FEW_SHOT_QUESTIONS + "\n"
    "QUAN TRONG:\n"
    "  - Ban PHAI su dung cac Predicate tu LOCAL ONTOLOGY duoc cung cap.\n"
    "  - question_type: 'yes_no' hoac 'multiple_choice'\n"
    "  - yes_no: chuyen statement thanh 1 AST node (statement_ast)\n"
    "  - multiple_choice: chuyen TUNG option A/B/C/D thanh AST (options_ast)\n\n"
    "Output JSON THUAN TUY:\n"
    '{\n'
    '  "step3_question_fol": [\n'
    '    {\n'
    '      "question_id": 0,\n'
    '      "question_type": "yes_no",\n'
    '      "source_nl": "statement to check",\n'
    '      "statement_ast": { <AST> }\n'
    '    }\n'
    '  ]\n'
    '}\n'
)

# Fallback
ANSWER_SYSTEM = (
    "Ban la chuyen gia suy luan logic. Dua vao cac tien de FOL da duoc Z3 xac minh, hay tra loi cau hoi.\n"
    "QUAN TRONG: Ban PHAI dua ra suy luan (reasoning) tung buoc TRUOC KHI dua ra dap an (answer).\n"
    'Tra loi JSON THUAN TUY: {"reasoning": "...", "answer": "A|B|C|D|Yes|No|Unknown"}\n'
)

# Physics
PHYSICS_SOLVER_SYSTEM = (
    "You are an expert physics problem solver. Solve step-by-step.\n"
    'Output PURE JSON: {"steps": ["..."], "answer": "<number>", "unit": "<unit>"}\n'
)
PHYSICS_MC_SYSTEM = (
    "You are an expert physics problem solver. Solve the multiple-choice problem.\n"
    "IMPORTANT: Write your step-by-step reasoning BEFORE giving the final answer.\n"
    'Output PURE JSON: {"reasoning": "...", "answer": "<answer>"}\n'
)

# ==================================================================
# UTILS
# ==================================================================
import os, csv, re, time
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path

def load_dataset(path, is_physics=False, max_samples=N_SAMPLES, split_mode=RUN_ON_SPLIT):
    if not path or not os.path.exists(path): return []
    if path.endswith(".csv"):
        with open(path, encoding="utf-8") as f: raw = list(csv.DictReader(f))
    else:
        with open(path, encoding="utf-8") as f: raw = json.load(f)
    
    out = raw[:max_samples]
    
    # --- SPLIT LOGIC ---
    import random
    rng = random.Random(SEED)
    rng.shuffle(out)
    n = len(out)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)
    
    if split_mode == "train":
        out = out[:n_train]
    elif split_mode == "val":
        out = out[n_train:n_train+n_val]
    elif split_mode == "test":
        out = out[n_train+n_val:]
    # if split_mode == "all", keep out as is

    if is_physics and out:
        for s in out:
            s["premises-NL"] = []
            s["questions"]   = [s.get("question", "")]
            s["answers"]     = [str(s.get("answer", "Unknown"))]
            s["_unit"]       = s.get("unit", "")
            s["_is_physics"] = True
    return out

def safe_json(text):
    text = text.strip()
    try: return json.loads(text)
    except: pass
    m = re.search(r"```(?:json)?\s*\n?(.*?)```", text, re.DOTALL)
    if m:
        try: return json.loads(m.group(1).strip())
        except: pass
    start = text.find("{")
    if start >= 0:
        depth, end = 0, start
        for i in range(start, len(text)):
            if text[i] == "{": depth += 1
            elif text[i] == "}":
                depth -= 1
                if depth == 0:
                    end = i
                    break
        try: return json.loads(text[start : end + 1])
        except: pass
    return {}

def batch_generate(prompt_pairs, max_tokens):
    formatted = []
    for sys_msg, usr_msg in prompt_pairs:
        messages = [{"role": "system", "content": sys_msg}, {"role": "user", "content": usr_msg}]
        formatted.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
    params = SamplingParams(temperature=0.05, max_tokens=max_tokens, repetition_penalty=1.1)
    outputs = llm.generate(formatted, params)
    outputs_sorted = sorted(outputs, key=lambda x: int(x.request_id))
    return [o.outputs[0].text.strip() for o in outputs_sorted]

def batch_generate_bon(prompt_pairs, max_tokens, n=BEST_OF_N):
    """Generate N candidates per prompt using higher temperature."""
    formatted = []
    for sys_msg, usr_msg in prompt_pairs:
        messages = [{"role": "system", "content": sys_msg},
                    {"role": "user", "content": usr_msg}]
        formatted.append(tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True))

    params = SamplingParams(
        temperature=BON_TEMPERATURE,
        max_tokens=max_tokens,
        repetition_penalty=1.1,
        n=n,
    )
    outputs = llm.generate(formatted, params)
    outputs_sorted = sorted(outputs, key=lambda x: int(x.request_id))

    return [
        [o.text.strip() for o in out.outputs]
        for out in outputs_sorted
    ]


def z3_select_best(candidates):
    """
    Try each candidate through Z3, pick the best one.
    Priority: sat > unsat > unknown > compile_error > no_ast
    """
    PRIORITY = {"sat": 4, "unsat": 3, "unknown": 2, "compile_error": 1, "no_ast": 0}
    best_data, best_status, best_score = {}, "no_ast", -1

    for raw in candidates:
        data = safe_json(raw)
        premises_ast = data.get("step2_premises_ast", [])

        if not premises_ast:
            score = PRIORITY["no_ast"]
            status = "no_ast"
        else:
            z3_info = verify_with_z3(premises_ast)
            status = z3_info["status"]
            score = PRIORITY.get(status, 0)

        if score > best_score:
            best_score = score
            best_status = status
            best_data = data

        if best_score >= 4:  # sat -> stop early
            break

    return best_data, best_status



In [ ]:

# ==================================================================
# STAGE 3 -- Z3 Compiler + Entailment Checker (deterministic, no AI)
# ==================================================================

_func_cache = {}


def get_z3_func(name, arity):
    key = f"{name}_{arity}"
    if key not in _func_cache:
        sorts = [IntSort()] * arity + [BoolSort()]
        _func_cache[key] = Function(name, *sorts)
    return _func_cache[key]


def _resolve_bound_var_name(bv):
    if isinstance(bv, dict):
        return bv.get("name", str(bv))
    return str(bv)


def _resolve_predicate_arg(a, var_map):
    if isinstance(a, str):
        if a in var_map:
            return var_map[a]
        return IntVal(abs(hash(a)) % 100000)
    if isinstance(a, dict):
        atype = a.get("type", "")
        name = a.get("name", "")
        if atype == "variable":
            if name in var_map:
                return var_map[name]
            v = Int(name)
            var_map[name] = v
            return v
        if atype == "constant":
            if name in var_map:
                return var_map[name]
            return IntVal(abs(hash(name)) % 100000)
        raise ValueError(f"Argument khong hop le (type={atype!r})")
    return IntVal(abs(hash(str(a))) % 100000)


def compile_ast(node, var_map):
    """Bien dich 1 AST node -> Z3 expression."""
    if not isinstance(node, dict):
        raise ValueError(f"Expected dict, got {type(node)}: {node!r}")

    ntype = node.get("type", "")

    if ntype == "quantifier":
        op = node.get("operator", "").lower()
        bvs = node.get("bound_variables", [])
        if not bvs:
            raise ValueError("quantifier thieu bound_variables")
        bv_names = [_resolve_bound_var_name(bv) for bv in bvs]
        z3_bvs = [Int(v) for v in bv_names]
        child_map = {**var_map, **{v: z3_bvs[i] for i, v in enumerate(bv_names)}}
        body = compile_ast(node["body"], child_map)
        if op == "forall":
            return ForAll(z3_bvs, body)
        elif op in ("exists", "exist"):
            return Exists(z3_bvs, body)
        else:
            raise ValueError(f"Quantifier khong hop le: {op!r}")

    elif ntype == "connective":
        op = node.get("operator", "").lower()
        ops = [compile_ast(o, var_map) for o in node.get("operands", [])]
        if op == "and":
            return And(*ops)
        elif op == "or":
            return Or(*ops)
        elif op == "implies":
            if len(ops) != 2:
                raise ValueError(f"implies can 2 operands, nhan {len(ops)}")
            return Implies(ops[0], ops[1])
        elif op == "iff":
            if len(ops) != 2:
                raise ValueError(f"iff can 2 operands, nhan {len(ops)}")
            return And(Implies(ops[0], ops[1]), Implies(ops[1], ops[0]))
        elif op == "not":
            if len(ops) != 1:
                raise ValueError(f"not can 1 operand, nhan {len(ops)}")
            return Not(ops[0])
        else:
            raise ValueError(f"Connective khong hop le: {op!r}")

    elif ntype == "predicate":
        name = node.get("name", "")
        args = node.get("arguments", [])
        if not name:
            raise ValueError('predicate thieu "name"')
        func = get_z3_func(name, len(args))
        z3_args = [_resolve_predicate_arg(a, var_map) for a in args]
        return func(*z3_args)

    elif ntype in ("variable", "constant"):
        name = node.get("name", "")
        if name in var_map:
            return var_map[name]
        if ntype == "constant":
            return IntVal(abs(hash(name)) % 100000)
        v = Int(name)
        var_map[name] = v
        return v

    else:
        raise ValueError(f"AST node type khong hop le: {ntype!r}")


def verify_with_z3(premises_ast):
    """Bien dich toan bo premises AST -> Z3, kiem tra consistency."""
    _func_cache.clear()
    solver = Solver()
    errors = []
    compiled = 0

    for item in premises_ast:
        pid = item.get("premise_id", "?")
        try:
            ast = item.get("ast", {})
            if not ast:
                errors.append(f"Premise {pid}: AST rong")
                continue
            expr = compile_ast(ast, {})
            solver.add(expr)
            compiled += 1
        except Exception as e:
            errors.append(f"Premise {pid}: {str(e)[:250]}")

    if errors:
        return {
            "status": "compile_error",
            "errors": errors,
            "compiled_count": compiled,
            "total_count": len(premises_ast),
        }
    try:
        result = solver.check()
        return {
            "status": str(result),
            "errors": [],
            "compiled_count": compiled,
            "total_count": len(premises_ast),
        }
    except Exception as e:
        return {
            "status": "solver_error",
            "errors": [str(e)],
            "compiled_count": compiled,
            "total_count": len(premises_ast),
        }


def hallucination_check(local_ontology, premises_ast):
    """Kiem tra: moi Predicate trong AST phai co trong Local Ontology."""
    declared = set()
    for item in local_ontology:
        if isinstance(item, dict):
            declared.add(item.get("predicate", ""))

    warnings = []

    def collect_predicates(node, found):
        if not isinstance(node, dict):
            return
        if node.get("type") == "predicate":
            found.add(node.get("name", ""))
        for v in node.values():
            if isinstance(v, dict):
                collect_predicates(v, found)
            elif isinstance(v, list):
                for sub in v:
                    collect_predicates(sub, found)

    for item in premises_ast:
        used = set()
        collect_predicates(item.get("ast", {}), used)
        hallucinated = used - declared - {""}
        if hallucinated:
            warnings.append(
                f"Premise {item.get('premise_id', '?')}: "
                f"predicates not in Ontology -> {hallucinated}"
            )
    return warnings


# ==================================================================
# Z3 ENTAILMENT CHECKER (NEW in v3)
# ==================================================================

def _compile_premises_to_solver(premises_ast):
    """Compile all premises into a Z3 solver. Return (solver, compiled, errors)."""
    _func_cache.clear()
    solver = Solver()
    solver.set("timeout", Z3_SOLVER_TIMEOUT)
    errors = []
    compiled = 0

    for item in premises_ast:
        pid = item.get("premise_id", "?")
        try:
            ast_node = item.get("ast", {})
            if not ast_node:
                continue
            expr = compile_ast(ast_node, {})
            solver.add(expr)
            compiled += 1
        except Exception as e:
            errors.append(f"P{pid}: {str(e)[:200]}")

    return solver, compiled, errors


def z3_entailment_check(premises_ast, question_fol_item):
    """
    Z3 entailment-based answer derivation for ONE question.

    Yes/No:
      YES:     premises + NOT(stmt) -> UNSAT  (stmt is entailed)
      NO:      premises + stmt -> UNSAT       (stmt contradicts)
      UNKNOWN: both SAT                       (undetermined)

    MC:
      Test each option for entailment, return the entailed one.

    Returns: {"answer": "...", "method": "...", "detail": "..."}
    """
    q_type = question_fol_item.get("question_type", "yes_no")

    if q_type == "yes_no":
        return _entail_yes_no(premises_ast, question_fol_item)
    elif q_type == "multiple_choice":
        return _entail_mc(premises_ast, question_fol_item)
    else:
        return {"answer": "Unknown", "method": "unsupported_qtype"}


def _entail_yes_no(premises_ast, q_item):
    """Entailment check for Yes/No/Unknown questions."""
    stmt_ast = q_item.get("statement_ast", {})
    if not stmt_ast:
        return {"answer": "Unknown", "method": "no_statement_ast"}

    # Compile the statement
    try:
        _func_cache_backup = dict(_func_cache)
        stmt_expr = compile_ast(stmt_ast, {})
    except Exception as e:
        return {"answer": "Unknown", "method": "stmt_compile_error",
                "detail": str(e)[:200]}

    # --- Test YES: premises + NOT(stmt) -> UNSAT? ---
    solver1, c1, e1 = _compile_premises_to_solver(premises_ast)
    if e1:
        return {"answer": "Unknown", "method": "premise_compile_error",
                "detail": "; ".join(e1[:3])}

    try:
        solver1.push()
        solver1.add(Not(stmt_expr))
        r1 = solver1.check()
        solver1.pop()
    except Exception as e:
        r1 = None

    if r1 == unsat:
        return {"answer": "Yes", "method": "z3_entailment",
                "detail": "premises + NOT(Q) is UNSAT => Q is entailed by premises"}

    # --- Test NO: premises + stmt -> UNSAT? ---
    solver2, _, _ = _compile_premises_to_solver(premises_ast)
    try:
        solver2.push()
        solver2.add(stmt_expr)
        r2 = solver2.check()
        solver2.pop()
    except Exception as e:
        r2 = None

    if r2 == unsat:
        return {"answer": "No", "method": "z3_negation",
                "detail": "premises + Q is UNSAT => Q contradicts premises"}

    # --- Neither -> Unknown ---
    return {"answer": "Unknown", "method": "z3_undetermined",
            "detail": "Neither Q nor NOT(Q) entailed"}


def _entail_mc(premises_ast, q_item):
    """Entailment check for Multiple Choice (A/B/C/D)."""
    options_ast = q_item.get("options_ast", {})
    if not options_ast:
        return {"answer": "Unknown", "method": "no_options_ast"}

    entailed = []
    consistent = []

    for label in ("A", "B", "C", "D"):
        opt_ast = options_ast.get(label, {})
        if not opt_ast:
            continue

        try:
            opt_expr = compile_ast(opt_ast, {})
        except:
            continue

        # Entailment: premises + NOT(option) -> UNSAT?
        solver, c, e = _compile_premises_to_solver(premises_ast)
        if e:
            continue

        try:
            solver.push()
            solver.add(Not(opt_expr))
            r = solver.check()
            solver.pop()

            if r == unsat:
                entailed.append(label)

            # Consistency: premises + option -> SAT?
            solver.push()
            solver.add(opt_expr)
            r2 = solver.check()
            solver.pop()
            if r2 == sat:
                consistent.append(label)
        except:
            continue

    if len(entailed) == 1:
        return {"answer": entailed[0], "method": "z3_unique_entailment",
                "detail": f"Only option {entailed[0]} is entailed"}
    elif len(entailed) > 1:
        return {"answer": entailed[0], "method": "z3_multi_entailment",
                "detail": f"Options {entailed} all entailed, picking first"}
    elif len(consistent) == 1:
        return {"answer": consistent[0], "method": "z3_unique_consistent",
                "detail": f"Only option {consistent[0]} is consistent with premises"}
    else:
        return {"answer": "Unknown", "method": "z3_mc_undetermined",
                "detail": f"Entailed={entailed}, Consistent={consistent}"}


print("Z3 compiler + entailment checker san sang")



In [ ]:

# ==================================================================
# STAGE 4 -- Batch Pipeline with Two-Pass Formalization
# ==================================================================

@dataclass
class PipelineResult:
    sample_id: int
    status: str = "pending"
    z3_status: str = "pending"
    z3_compiled: int = 0
    z3_total: int = 0
    z3_attempts: int = 0
    z3_errors: list = field(default_factory=list)
    hallucination_warn: list = field(default_factory=list)
    local_ontology: list = field(default_factory=list)
    premises_ast: list = field(default_factory=list)
    question_fol: list = field(default_factory=list)     
    predicted_answers: list = field(default_factory=list)
    ground_truth: list = field(default_factory=list)
    total_questions: int = 0
    correct_count: int = 0
    time_sec: float = 0.0
    error_log: list = field(default_factory=list)
    answer_source: list = field(default_factory=list)


# --- Prompt Builders ---
# ------------------------------------------------------------------
# SpaCy Entity Extraction (Direction D)
# ------------------------------------------------------------------
def extract_entities_spacy(premises_nl):
    """Extract entities from NL premises using SpaCy."""
    subjects, actions, objects = set(), set(), set()
    for premise in premises_nl:
        doc = nlp(premise)
        for ent in doc.ents:
            subjects.add(ent.text)
        for token in doc:
            if token.dep_ in ("nsubj", "nsubjpass") and token.pos_ in ("NOUN", "PROPN"):
                subjects.add(token.text)
            if token.pos_ == "VERB" and token.dep_ in ("ROOT", "relcl", "advcl", "xcomp", "ccomp"):
                actions.add(token.lemma_.capitalize())
            if token.dep_ in ("dobj", "pobj", "attr") and token.pos_ in ("NOUN", "PROPN"):
                objects.add(token.text)
            if token.pos_ == "ADJ" and token.dep_ in ("acomp", "attr"):
                actions.add("Is" + token.text.capitalize())
    return {"subjects": sorted(subjects), "actions": sorted(actions), "objects": sorted(objects)}


def _make_pass1_user(sample):
    premises = sample["premises-NL"]
    numbered_p = "\n".join(f"Premise {i+1}: {p}" for i, p in enumerate(premises))
    entities = extract_entities_spacy(premises)
    entity_hint = (
        "=== ENTITY HINTS (tu SpaCy -- HAY dung lam ten Predicate) ===\n"
        f"  Subjects  : {', '.join(entities['subjects']) or 'N/A'}\n"
        f"  Actions   : {', '.join(entities['actions']) or 'N/A'}\n"
        f"  Objects   : {', '.join(entities['objects']) or 'N/A'}\n"
    )
    return (
        f"Hay hinh thuc hoa cac tien de sau.\n\n"
        f"{entity_hint}\n"
        f"=== PREMISES ===\n{numbered_p}"
    )

def _make_pass1_correction_user(sample, result):
    premises = sample["premises-NL"]
    numbered_p = "\n".join(f"Premise {i+1}: {p}" for i, p in enumerate(premises))
    z3_errors = "\n".join(result.z3_errors[:5]) or "Loi khong xac dinh"
    prev_local = json.dumps(result.local_ontology, ensure_ascii=False, indent=2)
    return (
        f"Z3 Error:\n{z3_errors}\n\n"
        f"Ontology cu:\n{prev_local}\n\n"
        f"Premises:\n{numbered_p}\n\n"
        "Hay sua loi va tra ve JSON Buoc 1 + Buoc 2."
    )

def _make_pass2_user(sample, local_ontology):
    questions = sample.get("questions", [])
    numbered_q = "\n".join(f"Question {i+1}: {q}" for i, q in enumerate(questions))
    onto_str = json.dumps(local_ontology, ensure_ascii=False, indent=2)
    return (
        f"Su dung Local Ontology sau:\n{onto_str}\n\n"
        f"Hinh thuc hoa cac cau hoi sau de Z3 check entailment:\n"
        f"=== QUESTIONS ===\n{numbered_q}"
    )

def _make_answer_user(sample, fol_context, question, q_idx):
    p_text = "\n".join(f"P{i+1}: {p}" for i, p in enumerate(sample["premises-NL"]))
    fol_text = "\n".join(f"FOL P{i+1}: {f}" for i, f in enumerate(fol_context))
    return f"## Tien de NL:\n{p_text}\n\n## Tien de FOL:\n{fol_text}\n\n## Cau hoi {q_idx+1}:\n{question}"


# --- Parsers ---
def _process_pass1(result, raw_response):
    data = safe_json(raw_response)
    result.local_ontology = data.get("step1_local_ontology", [])
    result.premises_ast = data.get("step2_premises_ast", [])
    if not result.premises_ast:
        result.z3_status = "no_ast"
        return
    hw = hallucination_check(result.local_ontology, result.premises_ast)
    z3_info = verify_with_z3(result.premises_ast)
    result.hallucination_warn = hw
    result.z3_status = z3_info["status"]
    result.z3_errors = z3_info.get("errors", [])
    result.z3_compiled = z3_info.get("compiled_count", 0)
    result.z3_total = z3_info.get("total_count", 0)


# --- Physics (unchanged) ---
# (Keeping physics bypass exactly as it was)
def _normalize_number(s):
    s = str(s).strip()
    s = re.sub(r'\s*[A-Za-zμΩ°/%%]+\s*$', '', s).strip()
    m = re.match(r'^([+-]?[\d.]+)\s*[×xX\*]\s*10\s*\^?\s*\(?\s*([+-]?\d+)\s*\)?\s*$', s)
    if m:
        try: return float(m.group(1)) * (10 ** int(m.group(2)))
        except: pass
    try: return float(s)
    except: pass
    return None

def _physics_answer_match(predicted, ground_truth, tolerance=PHYSICS_TOLERANCE):
    p, g = str(predicted).strip(), str(ground_truth).strip()
    if p.lower() == g.lower(): return True
    p_num, g_num = _normalize_number(p), _normalize_number(g)
    if p_num is not None and g_num is not None:
        if g_num == 0: return abs(p_num) < 1e-9
        return abs(p_num - g_num) / abs(g_num) <= tolerance
    p_low, g_low = p.lower(), g.lower()
    if len(g_low) > 3 and (g_low in p_low or p_low in g_low): return True
    return False

def _run_physics_pipeline(samples, results, N, t0_all, dataset_name):
    print(f"\n[Physics Mode] Direct Solve -- {N} samples")
    solve_prompts = []
    for s in samples:
        q = s["questions"][0]
        u = s.get("_unit", "")
        if any(opt in q for opt in ["A.", "B.", "C.", "D.", "(A)", "(B)"]):
            solve_prompts.append((PHYSICS_MC_SYSTEM, f"Problem:\n{q}\n\nSolve and select the correct answer."))
        else:
            solve_prompts.append((PHYSICS_SOLVER_SYSTEM, f"Problem:\n{q}\nExpected unit: {u}\n\nSolve step-by-step."))
    responses = batch_generate(solve_prompts, PHYSICS_MAX_TOKENS)
    mc = 0
    for i, raw in enumerate(responses):
        ad = safe_json(raw)
        pred = str(ad.get("answer", "Unknown")).strip()
        results[i].predicted_answers.append({"question_id": 0, "answer": pred})
        results[i].z3_status, results[i].status = "skipped", "success"
        gt = samples[i].get("answers", [])
        if gt and _physics_answer_match(pred, gt[0]):
            results[i].correct_count = 1
            mc += 1
    print(f"Physics Match: {mc}/{N}")
    return results


# --- Main Pipeline ---
def run_batch_pipeline(samples, dataset_name="Dataset"):
    N = len(samples)
    t0_all = time.time()
    results = [PipelineResult(sample_id=i, ground_truth=s.get("answers", []), total_questions=len(s.get("questions", []))) for i, s in enumerate(samples)]

    if any(s.get("_is_physics", False) for s in samples):
        return _run_physics_pipeline(samples, results, N, t0_all, dataset_name)

    print(f"\n{'=' * 60}\n  PASS 1: FORMALIZE PREMISES (BoN={BEST_OF_N} + SpaCy)\n{'=' * 60}")
    t1 = time.time()
    form_prompts = [(PREMISE_FORMALIZATION_SYSTEM, _make_pass1_user(s)) for s in samples]
    all_candidates = batch_generate_bon(form_prompts, MAX_NEW_TOKENS_PASS1, n=BEST_OF_N)

    for i, candidates in enumerate(all_candidates):
        results[i].z3_attempts = 1
        try:
            best_data, best_status = z3_select_best(candidates)
            results[i].local_ontology = best_data.get("step1_local_ontology", [])
            results[i].premises_ast = best_data.get("step2_premises_ast", [])
            results[i].z3_status = best_status

            if results[i].premises_ast:
                hw = hallucination_check(results[i].local_ontology, results[i].premises_ast)
                results[i].hallucination_warn = hw
                z3_info = verify_with_z3(results[i].premises_ast)
                results[i].z3_errors = z3_info.get("errors", [])
                results[i].z3_compiled = z3_info.get("compiled_count", 0)
                results[i].z3_total = z3_info.get("total_count", 0)
        except Exception as e:
            results[i].z3_status = "no_ast"

    status_counts = Counter(r.z3_status for r in results)
    print(f"  Pass 1 Z3 Status (BoN={BEST_OF_N}): {dict(status_counts)}")

    # PASS 2: FORMALIZE QUESTIONS (Only for SAT/UNSAT)
    print(f"\n{'=' * 60}\n  PASS 2: FORMALIZE QUESTIONS\n{'=' * 60}")
    sat_indices = [i for i, r in enumerate(results) if r.z3_status in ("sat", "unsat", "unknown")]
    if sat_indices:
        pass2_prompts = [(QUESTION_FORMALIZATION_SYSTEM, _make_pass2_user(samples[i], results[i].local_ontology)) for i in sat_indices]
        pass2_responses = batch_generate(pass2_prompts, MAX_NEW_TOKENS_PASS2)
        for j, raw in enumerate(pass2_responses):
            idx = sat_indices[j]
            data = safe_json(raw)
            # Support both object and list return formats from LLM
            q_fol = data.get("step3_question_fol", [])
            if isinstance(q_fol, dict):
                q_fol = [q_fol]
            results[idx].question_fol = q_fol
        print(f"  Formalized questions for {len(sat_indices)} valid samples.")
    else:
        print("  No SAT samples for Pass 2.")

    # ENTAILMENT & FALLBACK
    print(f"\n{'=' * 60}\n  ENTAILMENT & FALLBACK\n{'=' * 60}")
    z3_ans, qw_fallbacks = 0, []
    
    for i, s in enumerate(samples):
        r = results[i]
        for q_idx, q in enumerate(s.get("questions", [])):
            answered_by_z3 = False
            if r.z3_status in ("sat", "unsat", "unknown") and r.question_fol:
                # Find matching question ID
                q_item = next((item for item in r.question_fol if isinstance(item, dict) and item.get("question_id") == q_idx), None)
                if not q_item and len(r.question_fol) > q_idx:
                    q_item = r.question_fol[q_idx]
                
                if q_item and isinstance(q_item, dict):
                    try:
                        ans_data = z3_entailment_check(r.premises_ast, q_item)
                        r.predicted_answers.append({
                            "question_id": q_idx, 
                            "answer": ans_data["answer"], 
                            "reasoning": f"[Z3] {ans_data.get('method','')}"
                        })
                        r.answer_source.append("z3")
                        z3_ans += 1
                        answered_by_z3 = True
                    except: pass
            
            if not answered_by_z3:
                qw_fallbacks.append((i, q_idx, q))
                r.answer_source.append("qwen")

    print(f"  Z3 Answers: {z3_ans} | Qwen Fallbacks: {len(qw_fallbacks)}")

    if qw_fallbacks:
        fb_prompts = []
        for i, q_idx, q in qw_fallbacks:
            ctx = [x.get("source_nl", "") for x in results[i].premises_ast]
            fb_prompts.append((ANSWER_SYSTEM, _make_answer_user(samples[i], ctx, q, q_idx)))
        fb_responses = batch_generate(fb_prompts, ANS_MAX_TOKENS)
        for j, raw in enumerate(fb_responses):
            i, q_idx, _ = qw_fallbacks[j]
            ans = safe_json(raw)
            results[i].predicted_answers.append({
                "question_id": q_idx, 
                "answer": ans.get("answer", "Unknown"),
                "reasoning": "[Qwen] fallback"
            })

    # Finalize scoring
    for i, r in enumerate(results):
        gt = samples[i].get("answers", [])
        correct = sum(1 for ar in r.predicted_answers if ar["question_id"] < len(gt) and str(ar["answer"]).strip().upper() == str(gt[ar["question_id"]]).strip().upper())
        r.correct_count = correct
        r.status = "success" if r.z3_status in ("sat", "unsat", "unknown") else ("partial" if r.z3_compiled > 0 else "failed")
        r.time_sec = round(time.time() - t0_all, 2)

    return results

print("Pipeline V5.2 (Best-of-N + SpaCy) Orchestrator loaded")


In [ ]:

# ==================================================================
# STAGE 5 -- Evaluation & Export
# ==================================================================

def evaluate(results):
    n = len(results)
    if n == 0:
        return {}
    total_q = sum(r.total_questions for r in results)
    total_ok = sum(r.correct_count for r in results)
    status_ct = {"success": 0, "partial": 0, "failed": 0}
    z3_ct = {"sat": 0, "unsat": 0, "unknown": 0, "compile_error": 0,
             "solver_error": 0, "no_ast": 0, "skipped": 0, "other": 0}
    for r in results:
        status_ct[r.status] = status_ct.get(r.status, 0) + 1
        key = r.z3_status if r.z3_status in z3_ct else "other"
        z3_ct[key] += 1
    hall_total = sum(len(r.hallucination_warn) for r in results)
    avg_retries = sum(r.z3_attempts for r in results) / n

    # Answer source breakdown
    z3_answers = sum(1 for r in results for s in r.answer_source if s == "z3")
    qwen_answers = sum(1 for r in results for s in r.answer_source if s == "qwen")

    # Z3 entailment accuracy (only Z3-answered questions)
    z3_correct = 0
    z3_total_q = 0
    qwen_correct = 0
    qwen_total_q = 0
    for r in results:
        gt = r.ground_truth
        for j, ar in enumerate(r.predicted_answers):
            qid = ar["question_id"]
            if qid >= len(gt):
                continue
            is_correct = str(ar["answer"]).strip().upper() == str(gt[qid]).strip().upper()
            src = r.answer_source[j] if j < len(r.answer_source) else "qwen"
            if src == "z3":
                z3_total_q += 1
                if is_correct:
                    z3_correct += 1
            else:
                qwen_total_q += 1
                if is_correct:
                    qwen_correct += 1

    return {
        "n_samples": n,
        "total_questions": total_q,
        "total_correct": total_ok,
        "accuracy": round(total_ok / total_q, 4) if total_q else 0,
        "status_breakdown": status_ct,
        "z3_breakdown": z3_ct,
        "hallucination_warnings": hall_total,
        "avg_z3_retries": round(avg_retries, 2),
        "answer_source": {
            "z3_entailment": z3_answers,
            "qwen_fallback": qwen_answers,
        },
        "z3_entailment_accuracy": round(z3_correct / z3_total_q, 4) if z3_total_q else 0,
        "z3_entailment_correct": z3_correct,
        "z3_entailment_total": z3_total_q,
        "qwen_accuracy": round(qwen_correct / qwen_total_q, 4) if qwen_total_q else 0,
        "qwen_correct": qwen_correct,
        "qwen_total": qwen_total_q,
    }


def result_to_dict(r):
    return {
        "sample_id": r.sample_id,
        "status": r.status,
        "z3_status": r.z3_status,
        "z3_compiled": r.z3_compiled,
        "z3_total": r.z3_total,
        "z3_attempts": r.z3_attempts,
        "z3_errors": r.z3_errors[:3],
        "hallucination_warns": r.hallucination_warn,
        "local_ontology": r.local_ontology,
        "correct_count": r.correct_count,
        "total_questions": r.total_questions,
        "predicted_answers": [a["answer"] for a in r.predicted_answers],
        "answer_sources": r.answer_source,
        "ground_truth": r.ground_truth,
        "per_question": [
            {
                "q_id": a["question_id"],
                "predicted": a["answer"],
                "gt": r.ground_truth[a["question_id"]] if a["question_id"] < len(r.ground_truth) else "?",
                "correct": (
                    str(a["answer"]).upper() == str(r.ground_truth[a["question_id"]]).upper()
                    if a["question_id"] < len(r.ground_truth) else False
                ),
                "reasoning": a.get("reasoning", ""),
                "source": r.answer_source[j] if j < len(r.answer_source) else "?",
            }
            for j, a in enumerate(r.predicted_answers)
        ],
        "time_sec": r.time_sec,
        "error_log": r.error_log[-2:],
    }


def finalize_and_save(results, output_path, dataset_path, dataset_name="Dataset"):
    if not results:
        print(f"[WARN] No results for {dataset_name}")
        return

    metrics = evaluate(results)
    acc_val = metrics.get("accuracy", 0)

    W = 65
    print("\n" + "=" * W)
    print(f"  {dataset_name.upper()} -- EVALUATION SUMMARY (v5.2 BoN+SpaCy)")
    print("=" * W)
    print(f"  Model          : {QWEN_MODEL_ID}")
    print(f"  Engine         : vLLM (TP={TENSOR_PARALLEL})")
    print(f"  Samples        : {metrics.get('n_samples', 0)}")
    print(f"  Total questions: {metrics.get('total_questions', 0)}")
    print(f"  Correct        : {metrics.get('total_correct', 0)}")
    print(f"  Accuracy       : {acc_val:.1%}")
    print("-" * W)
    print("  Pipeline Status:")
    for k, v in metrics.get("status_breakdown", {}).items():
        if v > 0:
            print(f"    {k:14}: {v:3d}  {'#' * min(v, 50)}")
    print("-" * W)
    print("  Z3 Verification:")
    for k, v in metrics.get("z3_breakdown", {}).items():
        if v > 0:
            print(f"    {k:16}: {v:3d}  {'#' * min(v, 50)}")
    print("-" * W)

    # Answer source breakdown
    src = metrics.get("answer_source", {})
    z3_a = src.get("z3_entailment", 0)
    qw_a = src.get("qwen_fallback", 0)
    print(f"  Answer Sources:")
    print(f"    Z3 entailment : {z3_a:>4} questions")
    print(f"    Qwen fallback : {qw_a:>4} questions")
    if z3_a > 0:
        z3_acc = metrics.get("z3_entailment_accuracy", 0)
        z3_c = metrics.get("z3_entailment_correct", 0)
        z3_t = metrics.get("z3_entailment_total", 0)
        print(f"  Z3 Entailment Accuracy: {z3_acc:.1%} ({z3_c}/{z3_t})")
    if qw_a > 0:
        qw_acc = metrics.get("qwen_accuracy", 0)
        qw_c = metrics.get("qwen_correct", 0)
        qw_t = metrics.get("qwen_total", 0)
        print(f"  Qwen Fallback Accuracy: {qw_acc:.1%} ({qw_c}/{qw_t})")
    print("-" * W)
    print(f"  Hallucination warns: {metrics.get('hallucination_warnings', 0)}")
    print(f"  Avg Z3 retries     : {metrics.get('avg_z3_retries', 0)}")
    print("=" * W)

    # Per-sample table
    hdr = f"{'ID':>3} | {'Status':>8} | {'Z3':>13} | {'Corr':>6} | {'Src':>7} | Hall"
    print(hdr)
    print("-" * len(hdr))
    show_n = min(len(results), 50)
    for r in results[:show_n]:
        hall = f"W{len(r.hallucination_warn)}" if r.hallucination_warn else "ok"
        src_str = "/".join(r.answer_source[:2]) if r.answer_source else "?"
        print(
            f"{r.sample_id:>3} | {r.status:>8} | {r.z3_status:>13} | "
            f"{r.correct_count}/{r.total_questions:>4} | {src_str:>7} | {hall}"
        )
    if len(results) > show_n:
        print(f"  ... ({len(results) - show_n} more)")

    # Save
    output_data = {
        "meta": {
            "model": QWEN_MODEL_ID,
            "engine": "vLLM",
            "tensor_parallel": TENSOR_PARALLEL,
            "pipeline_version": "v5.2_bon_spacy",
            "z3_entailment_enabled": Z3_ENTAILMENT,
            "n_samples": len(results),
            "max_retries": MAX_RETRIES,
            "dataset": dataset_path,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        },
        "metrics": metrics,
        "per_sample": [result_to_dict(r) for r in results],
    }
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(output_data, f, ensure_ascii=False, indent=2)

    fsize = Path(output_path).stat().st_size / 1024
    print(f"\nSaved: {output_path} ({fsize:.1f} KB)")
    print(f"Final Accuracy: {acc_val:.1%}  ({metrics.get('total_correct',0)}/{metrics.get('total_questions',0)})")


# ==================================================================
# RUN
# ==================================================================

print("\n" + "=" * 65)
print("  NEURO-SYMBOLIC PIPELINE v5.2 -- Best-of-N + SpaCy")
print(f"  Model: {QWEN_MODEL_ID}  |  TP: {TENSOR_PARALLEL} GPU(s)")
print(f"  Z3 Entailment: {Z3_ENTAILMENT}  |  Physics: {PHYSICS_MODE}")
print("=" * 65)

# --- Logic ---
logic_samples = load_dataset(DATASET_PATH, is_physics=False)
print(f"\nLogic Dataset: {len(logic_samples)} samples")
if logic_samples:
    logic_results = run_batch_pipeline(logic_samples, dataset_name="Logic Dataset")
    finalize_and_save(logic_results, OUTPUT_PATH, DATASET_PATH, "Logic Dataset")

# --- Physics ---
if PHYSICS_DATASET_PATH:
    physics_samples = load_dataset(PHYSICS_DATASET_PATH, is_physics=True)
    print(f"\nPhysics Dataset: {len(physics_samples)} samples")
    if physics_samples:
        po = OUTPUT_PATH.replace(".json", "_physics.json")
        physics_results = run_batch_pipeline(physics_samples, dataset_name="Physics Dataset")
        finalize_and_save(physics_results, po, PHYSICS_DATASET_PATH, "Physics Dataset")

print("\n" + "=" * 65)
print("  PIPELINE V5.2 HOAN TAT")
print("=" * 65)

